In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd, requests, os, time, json

ROOT=Path('/content/drive/MyDrive/US_ETF'); RES=ROOT/'model_lab_v1/results/open_revalidation_v1'; CACHE=ROOT/'directional_research/open_revalidation_1m_alpaca_v1/iex'
AUDIT=RES/'open_revalidation_trade_audit.parquet'; QUEUE=RES/'open_revalidation_targeted_backfill_queue_v1_7.csv'; OUT=RES/'targeted_backfill_v1_8_status_latest.csv'
print('='*100); print('KALMAN TARGETED ALPACA IEX BACKFILL v1.8 — RESEARCH ONLY'); print('='*100)

# Credentials: environment first, then Colab userdata. Values are never printed.
key=next((os.getenv(k) for k in ['APCA_API_KEY_ID','ALPACA_API_KEY','ALPACA_API_KEY_ID','APCA_API_KEY','ALPACA_KEY'] if os.getenv(k)),None)
sec=next((os.getenv(k) for k in ['APCA_API_SECRET_KEY','ALPACA_SECRET_KEY','ALPACA_API_SECRET_KEY','APCA_SECRET_KEY','ALPACA_SECRET'] if os.getenv(k)),None)
if not key or not sec:
    try:
        from google.colab import userdata
        for k in ['APCA_API_KEY_ID','ALPACA_API_KEY','ALPACA_API_KEY_ID','APCA_API_KEY','ALPACA_KEY']:
            try:
                v=userdata.get(k)
                if v: key=v; break
            except: pass
        for k in ['APCA_API_SECRET_KEY','ALPACA_SECRET_KEY','ALPACA_API_SECRET_KEY','APCA_SECRET_KEY','ALPACA_SECRET']:
            try:
                v=userdata.get(k)
                if v: sec=v; break
            except: pass
    except: pass
assert key and sec, 'Alpaca credentials not found in environment/Colab Secrets.'
print('[AUTH] credentials loaded (hidden)')

# IMPORTANT: v1.7 grouped widely separated trades into huge symbol windows. Rebuild a sparse per-trade queue here instead.
df=pd.read_parquet(AUDIT); df['entry_timestamp']=pd.to_datetime(df['entry_timestamp'],utc=True,errors='coerce'); df['exit_timestamp']=pd.to_datetime(df['exit_timestamp'],utc=True,errors='coerce')
common=['entry_price_iex','fixed4_exit_price_iex','open_0_price_iex','open_5_price_iex','open_15_price_iex']
mask=df[common].isna().any(axis=1) | df['prev_close_price_iex'].isna()
jobs=[]
for _,r in df.loc[mask].iterrows():
    if pd.isna(r.entry_timestamp): continue
    s=str(r.symbol).upper(); t=r.entry_timestamp; x=r.exit_timestamp if pd.notna(r.exit_timestamp) else t
    start=(t-pd.Timedelta(days=10 if pd.isna(r.prev_close_price_iex) else 2)).floor('D'); end=(max(t,x)+pd.Timedelta(days=2)).ceil('D')
    jobs.append([s,start,end])
jobs=pd.DataFrame(jobs,columns=['symbol','start','end']).drop_duplicates().sort_values(['symbol','start','end']).reset_index(drop=True)
print(f'[SPARSE QUEUE] jobs={len(jobs)} symbols={jobs.symbol.nunique()}')

headers={'APCA-API-KEY-ID':key,'APCA-API-SECRET-KEY':sec}; base='https://data.alpaca.markets/v2/stocks/bars'; status=[]
for i,r in jobs.iterrows():
    s=r.symbol; start=r.start.isoformat(); end=r.end.isoformat(); token=None; chunks=[]; pages=0; err=''
    try:
        while True:
            params={'symbols':s,'timeframe':'1Min','start':start,'end':end,'limit':10000,'adjustment':'raw','feed':'iex','sort':'asc'}
            if token: params['page_token']=token
            resp=requests.get(base,headers=headers,params=params,timeout=60)
            if resp.status_code==429: time.sleep(5); continue
            resp.raise_for_status(); js=resp.json(); pages+=1; arr=(js.get('bars') or {}).get(s,[])
            if arr: chunks.extend(arr)
            token=js.get('next_page_token')
            if not token: break
        if chunks:
            z=pd.DataFrame(chunks).rename(columns={'t':'timestamp','o':'open','h':'high','l':'low','c':'close','v':'volume','n':'trade_count','vw':'vwap'})
            z['timestamp']=pd.to_datetime(z['timestamp'],utc=True,errors='coerce'); z=z.dropna(subset=['timestamp']).drop_duplicates('timestamp').sort_values('timestamp')
            d=CACHE/s; d.mkdir(parents=True,exist_ok=True); path=d/f"{r.start.strftime('%Y-%m-%d')}__{r.end.strftime('%Y-%m-%d')}_targeted_v1_8.parquet"; z.to_parquet(path,index=False); st='DOWNLOADED'
        else: path=''; st='NO_DATA'
    except Exception as e: st='ERROR'; path=''; err=str(e)[:500]
    status.append({'symbol':s,'start':start,'end':end,'status':st,'rows':len(chunks),'pages':pages,'path':str(path),'error':err})
    if (i+1)%25==0 or i+1==len(jobs):
        pd.DataFrame(status).to_csv(OUT,index=False); vc=pd.Series([x['status'] for x in status]).value_counts().to_dict(); print(f'[{i+1}/{len(jobs)}] {vc}')
    time.sleep(0.12)

st=pd.DataFrame(status); st.to_csv(OUT,index=False)
print('\n[FINAL]'); print(st.status.value_counts(dropna=False).to_string()); print('rows_downloaded=',int(st.rows.sum())); print('status=',OUT)
print('research_only=True production_changed=False live_trading=False neon_write=False')
print('\nNEXT: rerun Audit Rebuild v1.6, then Checklist v1.5.')
